In [1]:
#  IMPORTS

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader, random_split, Subset
from torch.amp import autocast, GradScaler
import os
import numpy as np


#  DEVICE SETUP

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

torch.backends.cudnn.benchmark = True


#  HYPERPARAMETERS

BATCH_SIZE = 32
IMG_SIZE = 224
WARMUP_EPOCHS = 5
FINE_TUNE_EPOCHS = 25
NUM_WORKERS = 4
EARLY_STOPPING_PATIENCE = 6


#  DATA TRANSFORMS

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


#  LOAD DATASET 

DATA_DIR = r"C:\Users\Vaibhav\Desktop\Major\dataset"

base_dataset = datasets.ImageFolder(DATA_DIR)
num_classes = len(base_dataset.classes)

print("Classes:", base_dataset.classes)
print("Total images:", len(base_dataset))
print("Number of classes:", num_classes)

train_size = int(0.8 * len(base_dataset))
val_size = len(base_dataset) - train_size

indices = torch.randperm(len(base_dataset))
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(
    datasets.ImageFolder(DATA_DIR, transform=train_transform),
    train_indices
)

val_dataset = Subset(
    datasets.ImageFolder(DATA_DIR, transform=val_transform),
    val_indices
)

train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          num_workers=NUM_WORKERS,
                          pin_memory=True)

val_loader = DataLoader(val_dataset,
                        batch_size=BATCH_SIZE,
                        shuffle=False,
                        num_workers=NUM_WORKERS,
                        pin_memory=True)


#  LOAD RESNET50

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Replace classifier
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, num_classes)
)

model = model.to(DEVICE)


#  LOSS & AMP

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = GradScaler()


#  SAVE PATH

best_path = "models/best_resnet50.pth"
os.makedirs(os.path.dirname(best_path), exist_ok=True)


# TRAIN FUNCTION

def train_model(epochs, optimizer, scheduler=None):
    global best_val_acc
    patience_counter = 0

    for epoch in range(epochs):
        print(f"\nEpoch [{epoch+1}/{epochs}]")

        #  TRAIN 
        model.train()
        train_loss, correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            with autocast(device_type="cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        train_loss /= len(train_loader)

        #  VALIDATION
        model.eval()
        val_loss, correct, total = 0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)

                with autocast(device_type="cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_acc = 100 * correct / total
        val_loss /= len(val_loader)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

        if scheduler:
            scheduler.step()

        # SAVE BEST 
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)
            print("Best model saved!")
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break


#  STAGE 1 — WARMUP (ONLY FC)

print("\n Stage 1: Warmup")

best_val_acc = 0
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

train_model(WARMUP_EPOCHS, optimizer)


#  STAGE 2 — UNFREEZE LAYER4

print("\n Stage 2: Fine-Tune Layer4")

for param in model.layer4.parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 3e-4}
], weight_decay=5e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=FINE_TUNE_EPOCHS
)

train_model(FINE_TUNE_EPOCHS, optimizer, scheduler)

print("\nTraining Completed")
print("Best Validation Accuracy:", best_val_acc)
print("Best model saved at:", best_path)

Using device: cuda
Classes: ['0', '1', '2', '3', '4']
Total images: 10000
Number of classes: 5

🔵 Stage 1: Warmup

Epoch [1/5]
Train Loss: 1.2273 | Train Acc: 51.71%
Val Loss:   1.0824 | Val Acc:   60.90%
✅ Best model saved!

Epoch [2/5]
Train Loss: 1.0912 | Train Acc: 58.04%
Val Loss:   1.0275 | Val Acc:   62.65%
✅ Best model saved!

Epoch [3/5]
Train Loss: 1.0639 | Train Acc: 59.58%
Val Loss:   1.0175 | Val Acc:   62.70%
✅ Best model saved!

Epoch [4/5]
Train Loss: 1.0545 | Train Acc: 60.39%
Val Loss:   1.0061 | Val Acc:   62.80%
✅ Best model saved!

Epoch [5/5]
Train Loss: 1.0355 | Train Acc: 60.95%
Val Loss:   0.9988 | Val Acc:   63.15%
✅ Best model saved!

🔵 Stage 2: Fine-Tune Layer4

Epoch [1/25]
Train Loss: 0.9823 | Train Acc: 63.80%
Val Loss:   0.9077 | Val Acc:   67.35%
✅ Best model saved!

Epoch [2/25]
Train Loss: 0.8997 | Train Acc: 68.42%
Val Loss:   0.8413 | Val Acc:   70.75%
✅ Best model saved!

Epoch [3/25]
Train Loss: 0.8470 | Train Acc: 71.41%
Val Loss:   0.7989 | Val 

In [2]:

#  STAGE 3 — PERFORMANCE BOOST (5-CLASS OPTIMIZATION)

# Re-initialize the ResNet-50 architecture before loading weights
model = models.resnet50(weights=None) # We don't need ImageNet weights since we load ours
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, 5)
)
model = model.to(DEVICE)

# Now loading will work
model.load_state_dict(torch.load(best_path))

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

print("\n Computing Class Weights...")

# Get targets from full dataset
targets = [label for _, label in base_dataset.samples]
class_counts = np.bincount(targets)

class_weights = 1.0 / class_counts
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print("Class Counts:", class_counts)
print("Class Weights:", class_weights)

# Replace loss with weighted loss
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

print(" Weighted Loss Applied")



#  FINAL EVALUATION FUNCTION

def evaluate_model():
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            with autocast(device_type="cuda"):
                outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    print("\n Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))



#  RUN FINAL EVALUATION

print("\n Evaluating Best Model...")
model.load_state_dict(torch.load(best_path))
evaluate_model()

C:\Users\Vaibhav\AppData\Local\Temp\ipykernel_19616\541173985.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path))



🚀 Computing Class Weights...
Class Counts: [2000 2000 2000 2000 2000]
tensor([0.0005, 0.0005, 0.0005, 0.0005, 0.0005], device='cuda:0')
✅ Weighted Loss Applied

🔎 Evaluating Best Model...


C:\Users\Vaibhav\AppData\Local\Temp\ipykernel_19616\541173985.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path))



📊 Classification Report:
              precision    recall  f1-score   support

           0     0.9413    0.9901    0.9651       405
           1     0.9081    0.8442    0.8750       398
           2     0.7877    0.8179    0.8025       390
           3     0.8786    0.8913    0.8849       414
           4     0.8945    0.8626    0.8782       393

    accuracy                         0.8820      2000
   macro avg     0.8820    0.8812    0.8811      2000
weighted avg     0.8825    0.8820    0.8818      2000


📊 Confusion Matrix:
[[401   3   1   0   0]
 [ 19 336  36   2   5]
 [  5  23 319  24  19]
 [  0   1  28 369  16]
 [  1   7  21  25 339]]
